In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime
import json

In [ ]:
LATITUDE = -7.5412925
LONGITUDE = 110.4447503
INPUT_FILE = "semeru_dataset.csv"
OUTPUT_FILE = "semeru_dataset_with_wind_speed.csv"

In [ ]:
def parse_timestamp(timestamp_str):
    try:
        dt = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S %Z")
        date_str = dt.strftime("%Y-%m-%d")
        hour = dt.hour
        return date_str, hour
    except Exception as e:
        print(f"Error parsing timestamp {timestamp_str}: {e}")
        return None, None

In [ ]:
def get_wind_data(lat, lon, date_str, hour):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date_str,
        'end_date': date_str,
        'hourly': 'wind_speed_10m,wind_direction_10m',
        'wind_speed_unit': 'kmh',
        'timezone': 'UTC'
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            
            if 'hourly' in data:
                wind_speed = None
                wind_direction = None
                
                if 'wind_speed_10m' in data['hourly']:
                    speeds = data['hourly']['wind_speed_10m']
                    if hour < len(speeds) and speeds[hour] is not None:
                        wind_speed = round(speeds[hour], 2)
                
                if 'wind_direction_10m' in data['hourly']:
                    directions = data['hourly']['wind_direction_10m']
                    if hour < len(directions) and directions[hour] is not None:
                        wind_direction = round(directions[hour], 1)
                
                return wind_speed, wind_direction
            else:
                print(f"No hourly data available for {date_str}")
                return None, None
        else:
            print(f"API Error {response.status_code}: {response.text}")
            return None, None
    except Exception as e:
        print(f"Request failed: {e}")
        return None, None

In [ ]:
df = pd.read_csv(INPUT_FILE)
print(f"Total data: {len(df)} rows")
print(f"Processing with coordinates: Lat={LATITUDE}, Lon={LONGITUDE}")
print("-" * 60)

In [ ]:
print("=" * 60)
print("Processing wind data for empty values...")
print("=" * 60)

speed_filled = 0
direction_filled = 0
api_calls = 0
skipped = 0

for index, row in df.iterrows():
    current_speed = row['kec_angin_km_jam']
    current_direction = row['arah_angin_deg']
    
    speed_empty = pd.isna(current_speed) or current_speed == ''
    direction_empty = pd.isna(current_direction) or current_direction == ''
    
    if speed_empty or direction_empty:
        timestamp_str = row['timestamp']
        date_str, hour = parse_timestamp(timestamp_str)
        
        if date_str and hour is not None:
            print(f"Processing row {index + 1}/{len(df)}: {timestamp_str}")
            
            wind_speed, wind_direction = get_wind_data(LATITUDE, LONGITUDE, date_str, hour)
            
            if speed_empty and wind_speed is not None:
                df.at[index, 'kec_angin_km_jam'] = wind_speed
                speed_filled += 1
                print(f"  Wind speed: {wind_speed} km/h")
            
            if direction_empty and wind_direction is not None:
                df.at[index, 'arah_angin_deg'] = wind_direction
                direction_filled += 1
                print(f"  Wind direction: {wind_direction} deg")
            
            api_calls += 1
            time.sleep(0)
        else:
            print(f"Skipping row {index + 1}: Invalid timestamp")
            skipped += 1
        
        print()
    else:
        print(f"Row {index + 1}: Already complete (Speed: {current_speed} km/h, Direction: {current_direction} deg)")

print("=" * 60)
print(f"Processing Summary:")
print(f"Wind speeds filled: {speed_filled}")
print(f"Wind directions filled: {direction_filled}")
print(f"API calls made: {api_calls}")
print(f"Rows skipped (invalid timestamp): {skipped}")
print("=" * 60)

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print("=" * 60)
print(f"Data saved to: {OUTPUT_FILE}")
print(f"Total rows: {len(df)}")
print("=" * 60)

In [ ]:
print("Data preview:")
print(df[['timestamp', 'kec_angin_km_jam', 'arah_angin_deg']].head(10))